# 04 - From One Block to Real Text

## Imports

In [1]:
import warnings
# Same known benign numpy 2.x + Apple Accelerate BLAS quirk as ml_primer/cv_primer -
# spurious RuntimeWarnings on some matmuls with no actual NaN/Inf in the result.
warnings.filterwarnings('ignore', category=RuntimeWarning)

import numpy as np

# Make sure we get outputs from a probabilitistic model each time (for reproducibility)
np.random.seed(1515)

## What's Missing So Far

`03-transformer-block-and-history.ipynb` built one Transformer Block's forward pass and stopped - a sequence of vectors went in, a same-shaped sequence of vectors came out. That's genuinely most of a real model's architecture, stacked dozens of times over - but nothing about it produces *text*. Two pieces are still missing: turning that final sequence of vectors into an actual next-word guess, and then actually generating a sequence one word at a time.

## The Final Step: A Classification Head Over the Whole Vocabulary

After the last Transformer Block, a real model projects its final representation back onto the vocabulary - one logit per possible token, exactly the **Classification Head** from `deep_learning_primer/02-value-and-policy-networks.ipynb`, just with thousands of possible "classes" (every token in the vocabulary) instead of a handful of actions. Softmax turns those logits into a probability distribution over "what token comes next."

## A Note on What This Notebook Trains (and Doesn't)

Actually backpropagating through everything in `03` - attention's softmax, several stacked blocks, all of it - is real, valuable additional bookkeeping beyond what fits here. So for this notebook, the *next-token predictor* itself will be a simpler feedforward network - the exact same Classification Head shape and cross-entropy training already used in `deep_learning_primer/02`, `rl_primer/02`, and `perception_primer/07` - trained on a tiny, perfectly repeating toy corpus. What's real and unchanged from an actual Transformer is the **generation loop** itself: predict a distribution over the next token, pick one, feed it back in as part of the input for predicting the *next* token, repeat. That loop - not the specific network computing each step's prediction - is what "autoregressive" actually means.

## Training a Toy Next-Word Predictor

In [2]:
cycle = ["the", "robot", "climbs", "the", "wall", "then", "scores"]
corpus = cycle * 50   # the same short cycle, repeated many times

vocabulary = sorted(set(corpus))
word_to_id = {word: i for i, word in enumerate(vocabulary)}
vocab_size = len(vocabulary)
print(f"Vocabulary ({vocab_size} words): {vocabulary}")

CONTEXT_SIZE = 2   # predict the next word from the 2 words before it

def one_hot(word_id, size):
    vector = np.zeros(size)
    vector[word_id] = 1.0
    return vector

X, targets = [], []
for i in range(len(corpus) - CONTEXT_SIZE):
    context = corpus[i:i + CONTEXT_SIZE]
    target = corpus[i + CONTEXT_SIZE]
    X.append(np.concatenate([one_hot(word_to_id[w], vocab_size) for w in context]))
    targets.append(word_to_id[target])

X = np.array(X).T
n_examples = X.shape[1]
target_ids = np.array(targets)
Y_onehot = np.zeros((vocab_size, n_examples))
Y_onehot[target_ids, np.arange(n_examples)] = 1.0
print(f"Training examples: {n_examples}, input dim: {X.shape[0]}")

Vocabulary (6 words): ['climbs', 'robot', 'scores', 'the', 'then', 'wall']
Training examples: 348, input dim: 12


In [3]:
def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)

def softmax(z):
    exp_z = np.exp(z - np.max(z, axis=0, keepdims=True))
    return exp_z / np.sum(exp_z, axis=0, keepdims=True)

in_dim, hidden_dim, out_dim = X.shape[0], 16, vocab_size
W1 = np.random.randn(hidden_dim, in_dim) * (1 / np.sqrt(in_dim))
b1 = np.zeros((hidden_dim, 1))
W2 = np.random.randn(out_dim, hidden_dim) * (1 / np.sqrt(hidden_dim))   # Classification Head
b2 = np.zeros((out_dim, 1))

learning_rate = 0.1
for epoch in range(2000):
    z1 = W1 @ X + b1
    a1 = relu(z1)
    logits = W2 @ a1 + b2
    probabilities = softmax(logits)

    # the same cross-entropy gradient shape used for every Classification Head
    # so far: probabilities - true one-hot label
    d_logits = (probabilities - Y_onehot) / n_examples
    d_W2 = d_logits @ a1.T
    d_b2 = np.sum(d_logits, axis=1, keepdims=True)
    d_a1 = W2.T @ d_logits
    d_z1 = d_a1 * relu_derivative(z1)
    d_W1 = d_z1 @ X.T
    d_b1 = np.sum(d_z1, axis=1, keepdims=True)

    W1 -= learning_rate * d_W1
    b1 -= learning_rate * d_b1
    W2 -= learning_rate * d_W2
    b2 -= learning_rate * d_b2

final_predictions = np.argmax(softmax(W2 @ relu(W1 @ X + b1) + b2), axis=0)
accuracy = np.mean(final_predictions == target_ids)
print(f"Training accuracy: {accuracy:.2%}")

Training accuracy: 100.00%


## Autoregressive Generation

Now the actual loop: predict a distribution over the next word, choose one, append it to the sequence, and predict again - using the model's *own* last output as part of the input to its next prediction, exactly what `ai_resources/agent_primer` glossary term **Autoregressive Generation** means.

In [4]:
def predict_next_probabilities(context_words):
    context_vector = np.concatenate(
        [one_hot(word_to_id[w], vocab_size) for w in context_words]
    ).reshape(-1, 1)
    hidden = relu(W1 @ context_vector + b1)
    return softmax(W2 @ hidden + b2).flatten()

def generate(seed_words, n_steps, greedy=True, temperature=1.0, rng=None):
    generated = list(seed_words)
    for _ in range(n_steps):
        probabilities = predict_next_probabilities(generated[-CONTEXT_SIZE:])
        if greedy:
            next_id = np.argmax(probabilities)
        else:
            # temperature reshapes the distribution before sampling: > 1 flattens it
            # (more randomness), < 1 sharpens it (closer to greedy)
            scaled_logits = np.log(probabilities + 1e-12) / temperature
            scaled_probabilities = np.exp(scaled_logits - scaled_logits.max())
            scaled_probabilities /= scaled_probabilities.sum()
            next_id = rng.choice(len(vocabulary), p=scaled_probabilities)
        generated.append(vocabulary[next_id])
    return generated

greedy_output = generate(["the", "robot"], n_steps=14, greedy=True)
print("Greedy generation:", " ".join(greedy_output))

Greedy generation: the robot climbs the wall then scores the robot climbs the wall then scores the robot


With a perfectly deterministic training pattern and greedy decoding (always pick the single most likely next word), the model reproduces the cycle indefinitely - which is exactly correct here, not a failure. A real model generating from genuinely ambiguous text can get stuck the same way, greedily repeating itself, which is exactly why sampling strategies exist.

## Temperature: Trading Determinism for Variety

In [5]:
rng = np.random.default_rng(1515)
for temperature in [0.3, 1.5, 2.5]:
    output = generate(["the", "robot"], n_steps=14, greedy=False,
                       temperature=temperature, rng=np.random.default_rng(1515))
    print(f"Temperature={temperature}: {' '.join(output)}")

Temperature=0.3: the robot climbs the wall then scores the robot climbs the wall then scores the robot
Temperature=1.5: the robot climbs the wall then scores the robot climbs the wall then scores the robot
Temperature=2.5: the robot climbs the wall then the the climbs then the scores the robot climbs the


Notice temperatures 0.3 and 1.5 likely produced the *exact same* output as greedy decoding, not a gradually-more-random version of it. That's an honest result, not a bug: this toy model was trained on a perfectly deterministic pattern, so its learned probabilities are extremely close to 0 or 1 for every prediction - moderate temperature scaling barely dents a distribution that confident. Only a much higher temperature (2.5) has enough effect to actually flip a choice and visibly break the pattern. A real language model's training data is nowhere near this deterministic, so its per-token probabilities are naturally far less extreme, and temperature has a much more gradual, visible effect - but the mechanism you just built (rescale the logits, resample) is exactly the same either way.

## Why Chat Models Answer Questions Instead of Just Continuing Them

Everything built across this whole primer - tokenization, embeddings, attention, the Transformer Block, this generation loop - trains on one objective: **next-token prediction**. Given real text, predict whatever word is statistically likely to come next. That objective alone produces a model that *continues* text plausibly - if you feed it the start of a news article, it keeps writing a news article; if you feed it a question, its most likely continuation might just be... more questions, or a vaguely-related tangent, because that's a common pattern in raw training text too.

A model trained purely this way is not yet the chat assistant you're used to using in `ai_resources/ai_primer`. Turning it into one takes an additional training stage on top of next-token-prediction pretraining - commonly **instruction fine-tuning** (further training on examples that specifically look like `instruction -> helpful response` pairs) and **RLHF** (Reinforcement Learning from Human Feedback - literally the same Reinforcement Learning family from `ml_resources/rl_primer`, just with a reward signal built from human ratings of which responses are more helpful, rather than a scouting agent's match outcomes). This is why typing a question into a chat interface gets you an answer instead of a continuation of your question - that behavior was specifically trained into the model as a second stage, layered on top of the raw text-continuation skill this notebook actually built.

## Try It Yourself

Change `cycle` to a *non*-deterministic pattern - one where the same 2-word context can validly be followed by more than one different word (e.g. `["the", "robot", "climbs", "or", "scores", "the", "robot", "scores", "or", "climbs"]`, so `("the", "robot")` is sometimes followed by `climbs` and sometimes by `scores`). Retrain, check the new training accuracy (it should *not* reach 100% - explain in one sentence why a perfect accuracy is now impossible even for a perfectly trained model), and re-run temperature sampling at 0.3, 1.5, and 2.5. Does temperature start to matter at more moderate settings now, compared to the original deterministic cycle?

In [6]:
# TODO: build a non-deterministic cycle where one context can be followed by two
# different words, retrain, check accuracy (should not reach 100%), and re-run
# temperature sampling at 0.3, 1.5, 2.5 to see if it matters at lower settings now


## Resources

- Radford, A. et al. (2019). "Language Models are Unsupervised Multitask Learners" (the GPT-2 paper). Section 2 describes next-token prediction as the sole pretraining objective this notebook's generation loop is a simplified version of (paper).
- Ouyang, L. et al. (2022). "Training Language Models to Follow Instructions with Human Feedback" (the InstructGPT/RLHF paper). The real version of the fine-tuning stage described above (paper).
- [Hugging Face: How Do Transformers Work?](https://huggingface.co/learn/nlp-course/chapter1/4) - a concise overview connecting pretraining, fine-tuning, and generation the way this notebook did, with real production models as examples (instructional).